# Check saved models and reproduce recorded scores.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path

REVISION_ROOT = Path("revision_outputs")
DATA_DIR = Path("data")



PYTHON_EXE = str(RELEASE_CLASSICAL_PYTHON)


In [ ]:
import subprocess,sys,os,time,json,platform
from pathlib import Path
from datetime import datetime,timezone


def python_candidates(explicit=None):
    if explicit:
        return [str(Path(explicit).expanduser().absolute())]
    candidates=[Path(sys.executable)]
    bases={Path(sys.prefix),Path(sys.base_prefix),Path(sys.executable).parent.parent,
           Path('/opt/anaconda3'),Path('/opt/miniconda3'),Path.home()/'miniconda3',Path.home()/'anaconda3'}
    for base in sorted(bases):
        candidates.extend(sorted((base/'envs').glob('*/bin/python')))
    candidates.extend(sorted((Path.home()/'.conda/envs').glob('*/bin/python')))
    registry=Path.home()/'.conda/environments.txt'
    if registry.is_file():
        candidates.extend(Path(line.strip())/'bin/python' for line in registry.read_text().splitlines() if line.strip())
    for base in [Path.cwd(),Path.cwd().parent]:
        candidates.extend([base/'.venv/bin/python',base/'venv/bin/python'])

    unique=[]
    for path in candidates:
        value=str(path.expanduser().absolute())
        if path.is_file() and value not in unique:unique.append(value)
    return unique


def compatible_python(expected,out,explicit=None):
    code="""import json,platform,importlib.metadata as m
v={'python':platform.python_version()}
for key,package in [('numpy','numpy'),('pandas','pandas'),('sklearn','scikit-learn'),('tensorflow','tensorflow')]:
 try:v[key]=m.version(package)
 except m.PackageNotFoundError:v[key]='NOT_INSTALLED'
print(json.dumps(v))
"""
    reports=[]
    for executable in python_candidates(explicit):
        item={'executable':executable}
        try:
            proc=subprocess.run([executable,'-c',code],capture_output=True,text=True,timeout=20)
            item['metadata_returncode']=proc.returncode
            if proc.returncode!=0:
                item['status']='METADATA_FAILED';item['stderr_tail']=proc.stderr[-1500:];reports.append(item);continue
            versions=json.loads(proc.stdout.strip().splitlines()[-1]);item['versions']=versions
            if any(versions.get(k)!=v for k,v in expected.items()):
                item['status']='VERSIONS_DIFFER';reports.append(item);continue
            print('Eğitim sürümleriyle eşleşen Python kontrol ediliyor:',executable,flush=True)
            probe="import numpy, pandas; import tensorflow as tf; import sklearn; print('TF_PROBE_OK',float(tf.reduce_sum(tf.constant([1.,2.])).numpy()),flush=True)"
            proc=subprocess.run([executable,'-X','faulthandler','-c',probe],capture_output=True,text=True,timeout=60)
            item.update(probe_returncode=proc.returncode,probe_stdout_tail=proc.stdout[-1500:],probe_stderr_tail=proc.stderr[-3000:])
            if proc.returncode==0 and 'TF_PROBE_OK 3.0' in proc.stdout:
                item['status']='READY';reports.append(item);return executable,reports
            item['status']='IMPORT_OR_TENSOR_TEST_FAILED'
        except subprocess.TimeoutExpired:
            item['status']='PROBE_TIMEOUT'
        except (OSError,ValueError) as e:
            item.update(status='PROBE_ERROR',error=str(e))
        reports.append(item)
    return None,reports


def run_reproduction(root,data,config_template,worker_source,explicit_python=None):
    root=Path(root).expanduser().resolve();data=Path(data).expanduser().resolve()
    if not root.is_dir():raise FileNotFoundError('REVISION_ROOT yolu bulunamadı: '+str(root))
    out=root/'02C_saved_model_reproduction'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    out.mkdir(parents=True,exist_ok=False)
    config={**config_template,'revision_root':str(root),'data_directory':str(data),'output_directory':str(out)}
    cfgpath=out/'reproduction_config.json';cfgpath.write_text(json.dumps(config,ensure_ascii=False,indent=2))
    script=out/'reproduction_worker.py';script.write_text(worker_source)
    executable,probes=compatible_python(config['expected_versions'],out,explicit_python)
    reportpath=out/'model_reproduction_summary.json'
    base={'created_at_utc':datetime.now(timezone.utc).isoformat(),'parent_python':sys.executable,'environment_checks':probes,'source_recovery_audit_sha256':config['source_recovery_audit_sha256'],'AE_training_performed':False,'new_DBSCAN_or_OCSVM_results_produced':False}
    if executable is None:
        result={**base,'stage':'02C_FD001_saved_model_reproduction','status':'MATCHING_WORKING_ENVIRONMENT_NOT_FOUND','expected_versions':config['expected_versions']}
    else:
        print('Kullanılan Python:',executable,flush=True)
        logpath=out/'worker.log';started=time.monotonic();read_position=0;last_notice=started
        timed_out=False
        with logpath.open('w') as log:
            process=subprocess.Popen([executable,'-u','-X','faulthandler',str(script),str(cfgpath)],stdout=log,stderr=subprocess.STDOUT)
            try:
                while process.poll() is None:
                    time.sleep(1)
                    if time.monotonic()-started>1800:
                        process.kill();process.wait();timed_out=True;break
                    with logpath.open(errors='replace') as reader:
                        reader.seek(read_position);new=reader.read();read_position=reader.tell()
                    if new:print(new,end='',flush=True)
                    if time.monotonic()-last_notice>=30:
                        print('Kontrol sürüyor; geçen süre:',round(time.monotonic()-started),'sn',flush=True);last_notice=time.monotonic()
            except KeyboardInterrupt:
                process.terminate()
                try:process.wait(timeout=10)
                except subprocess.TimeoutExpired:process.kill();process.wait()
                raise
        with logpath.open(errors='replace') as reader:
            reader.seek(read_position);new=reader.read()
        if new:print(new,end='',flush=True)
        if process.returncode!=0 or not reportpath.is_file():
            result={**base,'stage':'02C_FD001_saved_model_reproduction','status':'WORKER_TIMEOUT' if timed_out else 'WORKER_PROCESS_FAILED','worker_returncode':process.returncode,'worker_log_tail':logpath.read_text(errors='replace')[-12000:]}
        else:
            result={**json.loads(reportpath.read_text()),**base,'worker_returncode':process.returncode,'worker_executable':executable,'elapsed_seconds':time.monotonic()-started}
    reportpath.write_text(json.dumps(result,ensure_ascii=False,indent=2,allow_nan=False))
    print('\nSON DURUM:',result['status'])
    print('Paylaşacağınız dosya:',reportpath.resolve())
    return result,reportpath


In [ ]:
CONFIG_TEMPLATE = load_json('assets/02C/04_config_template.json')

WORKER_SOURCE = load_text('assets/02C/04_worker_source.py')

result, summary_path = run_reproduction(REVISION_ROOT, DATA_DIR, CONFIG_TEMPLATE, WORKER_SOURCE, PYTHON_EXE)
